# Scrap SidneyKim0

YouTube 비디오 분석 + 커뮤니티 게시글 수집 → `Analysis_sidneykim0.db`

- **videos**: Gemini API로 영상 슬라이드 분석
- **community**: Playwright로 커뮤니티 게시글 + 이미지 분석

## 1. Configuration

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# ── Target ────────────────────────────────────────────────────────────────────
CHANNEL = "sidneykim0"

# ── .env 로드 ─────────────────────────────────────────────────────────────────
env_path = Path(os.path.dirname(os.path.abspath('__file__'))) / ".env"
load_dotenv(dotenv_path=env_path)

# ── DB 경로 ───────────────────────────────────────────────────────────────────
_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
DB_DIR = os.path.normpath(os.path.join(
    _notebook_dir, '..', 'Personas', 'SidneyKim0'))
DB_PATH = os.path.join(DB_DIR, 'Analysis_sidneykim0.db')
os.makedirs(DB_DIR, exist_ok=True)

# ── 시스템 프롬프트 경로 ───────────────────────────────────────────────────────
PROMPT_PATH = os.path.normpath(os.path.join(
    _notebook_dir, '..', '..', '..', '..',
    'References', 'Scrap_Youtube', 'prompt_sidneykim0.md'))

# ── Options ───────────────────────────────────────────────────────────────────
MAX_POSTS = 1000       # 커뮤니티 최대 게시물 수
WORKERS = 50           # 병렬 워커 수
HEADLESS = True        # 브라우저 창 숨기기
SINCE = None           # 비디오 날짜 필터 (YYYY-MM-DD 또는 None)
MAX_COUNT = None       # 비디오 최대 처리 수 (None = 전체)

print(f"DB path: {DB_PATH}")
print(f"Prompt path: {PROMPT_PATH}")
print(f"GEMINI_API_KEY: {'set' if os.environ.get('GEMINI_API_KEY') else 'NOT SET'}")

## 2. Setup

### 2.1 Imports & Constants
외부 라이브러리 임포트 및 이미지 분석 프롬프트 상수

In [ ]:
# ── Imports & Constants ────────────────────────────────────────────────

import asyncio
import json
import re
import sqlite3
import subprocess
import tempfile
import threading
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta

from google import genai
from google.genai import types
from playwright.async_api import async_playwright
from tqdm.auto import tqdm

IMAGE_ANALYSIS_PROMPT = """이미지에서 데이터와 의미를 추출하세요. 게시글 본문에 이미 있는 정보는 반복하지 마세요.

게시글 본문:
{content_text}

규칙:
- 서두/인사말 없이 바로 내용 시작
- 색상, 위치 등 시각적 묘사 생략. 데이터와 의미만 추출
- 차트/그래프: 축 레이블, 구체적 수치, 데이터 포인트, 추세 방향과 변곡점
- 표: 모든 행/열 데이터를 그대로 전사
- 텍스트: 있는 그대로 전사
- 스크린샷: 핵심 정보(수치, 상태, 결과)만 추출
- 게시글 본문과 중복되는 내용은 제외"""

### 2.2 Utilities
한국어 상대시간 파싱, 시스템 프롬프트 로드, 비디오 URL 파싱 등 공통 헬퍼

In [ ]:
# ── 유틸리티 함수 ──────────────────────────────────────────────────────


def parse_relative_time_korean(relative_str: str) -> str:
    cleaned = re.sub(r"\s*\(수정됨\)\s*", "", relative_str).strip()
    match = re.search(r"(\d+)\s*(시간|일|주|개월|년)\s*전", cleaned)
    if not match:
        return datetime.now().strftime("%Y-%m-%d")
    amount = int(match.group(1))
    unit = match.group(2)
    unit_map = {"시간": "hours", "일": "days", "주": "weeks", "개월": "months", "년": "years"}
    key = unit_map.get(unit, "days")
    if key == "hours":
        delta = timedelta(hours=amount)
    elif key == "days":
        delta = timedelta(days=amount)
    elif key == "weeks":
        delta = timedelta(weeks=amount)
    elif key == "months":
        delta = timedelta(days=amount * 30)
    elif key == "years":
        delta = timedelta(days=amount * 365)
    else:
        delta = timedelta(days=amount)
    return (datetime.now() - delta).strftime("%Y-%m-%d")


def load_system_prompt(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def extract_video_id(url: str) -> str:
    if "v=" in url:
        return url.split("v=")[1].split("&")[0]
    elif "youtu.be/" in url:
        return url.split("youtu.be/")[1].split("?")[0]
    return "unknown"


def is_single_video(url: str) -> bool:
    if "list=" in url and "watch?v=" not in url:
        return False
    if "/@" in url or "/channel/" in url or "/c/" in url:
        return False
    if "watch?v=" in url or "youtu.be/" in url:
        return True
    return False

### 2.3 Database Layer
SQLite 초기화, 중복 확인, videos/community 테이블 CRUD

In [ ]:
# ── 데이터베이스 레이어 ────────────────────────────────────────────────


def init_db() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS videos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            video_id TEXT NOT NULL UNIQUE,
            date_upload TEXT NOT NULL,
            title TEXT NOT NULL,
            url TEXT NOT NULL UNIQUE,
            slides TEXT NOT NULL,
            scraped_at TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_videos_date ON videos(date_upload);
        CREATE INDEX IF NOT EXISTS idx_videos_video_id ON videos(video_id);

        CREATE TABLE IF NOT EXISTS community (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            post_id TEXT NOT NULL UNIQUE,
            date_publish TEXT NOT NULL,
            content_text TEXT NOT NULL,
            content_image TEXT,
            scraped_at TEXT NOT NULL
        );
        CREATE INDEX IF NOT EXISTS idx_community_date ON community(date_publish);
        CREATE INDEX IF NOT EXISTS idx_community_post_id ON community(post_id);
        """
    )
    conn.commit()
    return conn


def get_existing_video_urls(conn: sqlite3.Connection) -> set[str]:
    rows = conn.execute("SELECT url FROM videos").fetchall()
    return {r[0] for r in rows}


def get_existing_post_ids(conn: sqlite3.Connection) -> set[str]:
    rows = conn.execute("SELECT post_id FROM community").fetchall()
    return {r[0] for r in rows}


def insert_video(conn: sqlite3.Connection, video_id: str, date_upload: str,
                 title: str, url: str, slides: str, scraped_at: str,
                 db_lock: threading.Lock) -> None:
    with db_lock:
        conn.execute(
            "INSERT OR IGNORE INTO videos (video_id, date_upload, title, url, slides, scraped_at) "
            "VALUES (?, ?, ?, ?, ?, ?)",
            (video_id, date_upload, title, url, slides, scraped_at),
        )
        conn.commit()


def insert_community_post(conn: sqlite3.Connection, post_id: str, date_publish: str,
                          content_text: str, content_image: str | None,
                          scraped_at: str) -> None:
    conn.execute(
        "INSERT OR IGNORE INTO community (post_id, date_publish, content_text, content_image, scraped_at) "
        "VALUES (?, ?, ?, ?, ?)",
        (post_id, date_publish, content_text, content_image, scraped_at),
    )
    conn.commit()

### 2.4 Video Pipeline
yt-dlp 메타데이터 수집 → Gemini API 슬라이드 분석 → DB 저장 (병렬)

In [ ]:
# ── 비디오 파이프라인 ──────────────────────────────────────────────────


def get_video_list(channel_url: str) -> list[dict]:
    result = subprocess.run(
        ["yt-dlp", "--flat-playlist", "-j", channel_url],
        capture_output=True, text=True, timeout=120,
    )
    if result.returncode != 0:
        print(f"yt-dlp 오류: {result.stderr.strip()}")
        return []

    channel_from_url = ""
    if "/@" in channel_url:
        channel_from_url = channel_url.split("/@")[1].split("/")[0]

    videos = []
    for line in result.stdout.strip().split("\n"):
        if not line:
            continue
        entry = json.loads(line)
        videos.append({
            "id": entry.get("id", "unknown"),
            "title": entry.get("title", ""),
            "channel": entry.get("channel") or entry.get("uploader") or channel_from_url,
            "upload_date": entry.get("upload_date", ""),
            "url": f"https://www.youtube.com/watch?v={entry['id']}",
        })
    return videos


def filter_videos(videos: list[dict], max_count: int | None = None,
                  since: str | None = None) -> list[dict]:
    if since:
        since_date = since.replace("-", "")
        videos = [v for v in videos if v["upload_date"] >= since_date]
    if max_count:
        videos = videos[:max_count]
    return videos


def analyze_video(youtube_url: str, system_prompt: str, retries: int = 3) -> dict | None:
    api_key = os.environ.get("GEMINI_API_KEY")
    model_name = os.environ.get("GEMINI_MODEL", "gemini-3-flash-preview")
    client = genai.Client(api_key=api_key)

    for attempt in range(1, retries + 1):
        try:
            response = client.models.generate_content(
                model=f"models/{model_name}",
                contents=types.Content(
                    parts=[
                        types.Part(file_data=types.FileData(file_uri=youtube_url)),
                        types.Part(text=system_prompt),
                    ]
                ),
            )
            text = response.text.strip()
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            return json.loads(text.strip())
        except Exception as e:
            error_msg = str(e)
            if "INVALID_ARGUMENT" in error_msg and "token" in error_msg.lower():
                return None
            if attempt < retries:
                wait = 2 ** attempt
                print(f"  재시도 {attempt}/{retries} ({wait}s 대기)...")
                time.sleep(wait)
            else:
                print(f"  {retries}회 시도 후 실패: {e}")
                return None
    return None


def fetch_single_metadata(video_id: str) -> dict:
    try:
        result = subprocess.run(
            ["yt-dlp", "--no-download", "--print",
             "%(title)s\t%(upload_date)s\t%(channel)s",
             f"https://www.youtube.com/watch?v={video_id}"],
            capture_output=True, text=True, timeout=30,
        )
        if result.returncode == 0 and result.stdout.strip():
            parts = result.stdout.strip().split("\t")
            if len(parts) >= 3:
                return {"title": parts[0], "upload_date": parts[1], "channel": parts[2]}
    except Exception:
        pass
    return {}


def process_single_video(video_info: dict, system_prompt: str,
                         conn: sqlite3.Connection, db_lock: threading.Lock) -> dict:
    vid = video_info["id"]
    url = video_info["url"]
    title = video_info.get("title", "")

    if not video_info.get("upload_date") or not video_info.get("channel"):
        meta = fetch_single_metadata(vid)
        if meta:
            video_info["title"] = meta.get("title") or title
            video_info["upload_date"] = meta.get("upload_date") or video_info.get("upload_date", "")
            video_info["channel"] = meta.get("channel") or video_info.get("channel", "")
            title = video_info["title"]

    print(f"[{vid}] 분석 시작: {title or url}")
    result = analyze_video(url, system_prompt)

    if result is None:
        print(f"[{vid}] 스킵됨 (토큰 초과 또는 API 에러)")
        return {"id": vid, "title": title, "url": url, "status": "skipped"}

    upload_date = video_info.get("upload_date", "")
    if upload_date and len(upload_date) == 8:
        upload_date = f"{upload_date[:4]}-{upload_date[4:6]}-{upload_date[6:8]}"

    slides = result.get("slides", result.get("transcript", []))
    slides_json = json.dumps(slides, ensure_ascii=False)
    scraped_at = datetime.now().isoformat(timespec="seconds")

    insert_video(conn, vid, upload_date, title, url, slides_json, scraped_at, db_lock)

    slide_count = len(slides)
    print(f"[{vid}] 완료: {slide_count}개 슬라이드 → DB 저장")
    return {"id": vid, "title": title, "url": url, "status": "success"}


def run_videos(conn: sqlite3.Connection, db_lock: threading.Lock) -> None:
    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("오류: GEMINI_API_KEY 환경 변수가 설정되지 않았습니다.")
        return

    system_prompt = load_system_prompt(PROMPT_PATH)
    channel_url = f"https://www.youtube.com/@{CHANNEL}"

    print("영상 목록 추출 중 (yt-dlp --flat-playlist)...")
    videos = get_video_list(channel_url)
    print(f"총 {len(videos)}개 영상 발견")

    videos = filter_videos(videos, max_count=MAX_COUNT, since=SINCE)
    print(f"필터 적용 후 {len(videos)}개 대상")

    if not videos:
        print("처리할 영상이 없습니다.")
        return

    existing_urls = get_existing_video_urls(conn)
    videos = [v for v in videos if v["url"] not in existing_urls]
    print(f"신규 영상: {len(videos)}개 (기존 {len(existing_urls)}개 제외)")

    if not videos:
        print("저장할 신규 영상이 없습니다.")
        return

    videos.sort(key=lambda v: v.get("upload_date", ""))

    results = []
    success_count = 0
    skipped_count = 0

    print(f"\n{len(videos)}개 영상을 {WORKERS}개 워커로 병렬 처리합니다.\n")
    with ThreadPoolExecutor(max_workers=WORKERS) as executor:
        futures = {
            executor.submit(process_single_video, v, system_prompt, conn, db_lock): v
            for v in videos
        }
        with tqdm(total=len(videos), desc="영상 분석", unit="개") as pbar:
            for future in as_completed(futures):
                result = future.result()
                results.append(result)
                if result["status"] == "success":
                    success_count += 1
                else:
                    skipped_count += 1
                pbar.set_postfix(success=success_count, skipped=skipped_count)
                pbar.update(1)

    success = sum(1 for r in results if r["status"] == "success")
    skipped = sum(1 for r in results if r["status"] == "skipped")
    print(f"\n{'=' * 50}")
    print(f"비디오 완료: {success}건 성공, {skipped}건 스킵 (총 {len(results)}건)")

    if skipped > 0:
        print("\n스킵된 영상:")
        for r in results:
            if r["status"] == "skipped":
                print(f"  - {r['title'] or r['id']} ({r['url']})")

### 2.5 Community Pipeline
Playwright 커뮤니티 크롤링 → Gemini 이미지 분석 → DB 저장 (병렬)

In [ ]:
# ── 커뮤니티 파이프라인 ────────────────────────────────────────────────


async def crawl_youtube_community(channel: str, max_posts: int = 1000,
                                  headless: bool = True) -> list[dict]:
    channel_url = f"https://www.youtube.com/@{channel}/posts"
    print(f"'{channel}' 채널 커뮤니티 크롤링 시작 (목표: {max_posts}개)\n")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=headless)
        page = await browser.new_page()

        try:
            print("페이지 로딩 중...")
            await page.goto(channel_url, wait_until="domcontentloaded", timeout=60000)
            await page.wait_for_selector("ytd-backstage-post-thread-renderer", timeout=30000)
            print("페이지 로드 완료\n")
        except Exception as e:
            print(f"오류: 페이지 로드 실패 - {e}")
            await browser.close()
            return []

        posts_data = []
        crawled_post_ids: set[str] = set()
        no_new_posts_streak = 0
        max_scroll_attempts = max(20, max_posts // 5)
        scroll_count = 0

        pbar = tqdm(total=max_posts, desc="게시물 수집", unit="개")

        while len(posts_data) < max_posts and scroll_count < max_scroll_attempts:
            initial_count = len(posts_data)
            post_elements = await page.query_selector_all("ytd-backstage-post-thread-renderer")

            for element in post_elements:
                if len(posts_data) >= max_posts:
                    break

                time_link = await element.query_selector("#published-time-text a")
                if not time_link:
                    continue

                post_url = await time_link.get_attribute("href")
                if not post_url or "/post/" not in post_url:
                    continue

                post_id = post_url.split("/post/")[1]
                if post_id in crawled_post_ids:
                    continue
                crawled_post_ids.add(post_id)

                try:
                    content_elem = await element.query_selector("#content-text")
                    content = await content_elem.inner_text() if content_elem else ""

                    image_url = ""
                    image_elem = await element.query_selector("ytd-backstage-image-renderer img")
                    if not image_elem:
                        image_elem = await element.query_selector("#image-container img")
                    if image_elem:
                        await image_elem.scroll_into_view_if_needed()
                        await page.wait_for_timeout(500)
                        image_url = (
                            await image_elem.get_attribute("src")
                            or await image_elem.get_attribute("data-src")
                            or ""
                        )

                    published_time = await time_link.inner_text()

                    posts_data.append({
                        "post_id": post_id,
                        "content": content.strip(),
                        "image_url": image_url,
                        "published_time": published_time.strip(),
                    })
                    pbar.update(1)
                except Exception as e:
                    pbar.write(f"데이터 추출 실패: {e}")
                    continue

            if len(posts_data) >= max_posts:
                pbar.write(f"목표 수량({max_posts}개)에 도달!")
                break

            if len(posts_data) == initial_count:
                no_new_posts_streak += 1
                if no_new_posts_streak >= 3:
                    pbar.write("3회 연속 새 게시물 없음 - 종료")
                    break
            else:
                no_new_posts_streak = 0

            if len(posts_data) < max_posts:
                scroll_count += 1
                pbar.write(f"스크롤 중... ({scroll_count}/{max_scroll_attempts})")
                await page.evaluate("window.scrollBy(0, window.innerHeight * 2)")
                await page.wait_for_timeout(2500)

        pbar.close()
        await browser.close()

    return posts_data


def download_image(image_url: str, dest_path: str) -> bool:
    try:
        urllib.request.urlretrieve(image_url, dest_path)
        return True
    except Exception as e:
        print(f"  이미지 다운로드 실패: {e}")
        return False


def analyze_community_image(image_path: str, content_text: str, retries: int = 3) -> str | None:
    api_key = os.environ.get("GEMINI_API_KEY")
    model_name = os.environ.get("GEMINI_MODEL", "gemini-3-flash-preview")
    client = genai.Client(api_key=api_key)
    prompt = IMAGE_ANALYSIS_PROMPT.format(content_text=content_text)

    for attempt in range(1, retries + 1):
        try:
            with open(image_path, "rb") as f:
                image_bytes = f.read()
            ext = os.path.splitext(image_path)[1].lower()
            mime_map = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png",
                        ".webp": "image/webp", ".gif": "image/gif"}
            mime_type = mime_map.get(ext, "image/jpeg")
            response = client.models.generate_content(
                model=f"models/{model_name}",
                contents=types.Content(
                    parts=[
                        types.Part(inline_data=types.Blob(mime_type=mime_type, data=image_bytes)),
                        types.Part(text=prompt),
                    ]
                ),
            )
            return response.text.strip()
        except Exception as e:
            if attempt < retries:
                wait = 2 ** attempt
                time.sleep(wait)
            else:
                print(f"  이미지 분석 실패: {e}")
                return None
    return None


def process_single_post(post: dict, temp_dir: str,
                        conn: sqlite3.Connection, db_lock: threading.Lock) -> dict:
    post_id = post["post_id"]
    image_url = post.get("image_url", "")
    content_text = post.get("content", "")
    content_image = None

    if image_url:
        image_path = os.path.join(temp_dir, f"{post_id}.jpg")
        if download_image(image_url, image_path):
            try:
                content_image = analyze_community_image(image_path, content_text)
            finally:
                if os.path.exists(image_path):
                    os.remove(image_path)

    scraped_at = datetime.now().isoformat(timespec="seconds")
    with db_lock:
        insert_community_post(conn, post_id, post["date_publish"],
                              content_text, content_image, scraped_at)

    return {"post_id": post_id, "status": "success"}


async def run_community(conn: sqlite3.Connection, db_lock: threading.Lock) -> None:
    posts = await crawl_youtube_community(CHANNEL, MAX_POSTS, HEADLESS)
    if not posts:
        print("수집된 게시물이 없습니다.")
        return

    print(f"\n총 {len(posts)}개 게시물 수집 완료")

    existing_ids = get_existing_post_ids(conn)
    posts = [p for p in posts if p["post_id"] not in existing_ids]
    print(f"신규 게시물: {len(posts)}개 (기존 {len(existing_ids)}개 제외)")

    if not posts:
        print("저장할 신규 게시물이 없습니다.")
        return

    posts.reverse()

    for post in posts:
        post["date_publish"] = parse_relative_time_korean(post.get("published_time", ""))

    has_images = any(p.get("image_url") for p in posts)
    if has_images and not os.environ.get("GEMINI_API_KEY"):
        print("경고: GEMINI_API_KEY가 없어 이미지 분석을 건너뜁니다.")

    print(f"\n게시물 처리 중... ({len(posts)}개)")
    saved = 0
    with tempfile.TemporaryDirectory() as temp_dir:
        with ThreadPoolExecutor(max_workers=WORKERS) as executor:
            futures = {
                executor.submit(process_single_post, p, temp_dir, conn, db_lock): p["post_id"]
                for p in posts
            }
            with tqdm(total=len(posts), desc="게시물 처리+저장", unit="개") as pbar:
                for future in as_completed(futures):
                    future.result()
                    saved += 1
                    pbar.update(1)

    print(f"\n{'=' * 50}")
    print(f"커뮤니티 완료: {saved}건 저장")


print("Setup complete.")

## 3. Scrape

In [ ]:
conn = init_db()
db_lock = threading.Lock()

try:
    print("=== 커뮤니티 게시글 수집 ===\n")
    await run_community(conn, db_lock)
    print(f"\n\n=== 비디오 분석 ===\n")
    run_videos(conn, db_lock)
finally:
    conn.close()

## 4. Explore DB

In [ ]:
# ── DB 요약 ─────────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)

video_count = conn.execute("SELECT COUNT(*) FROM videos").fetchone()[0]
community_count = conn.execute("SELECT COUNT(*) FROM community").fetchone()[0]
print(f"Videos: {video_count}")
print(f"Community: {community_count}")
print(f"Total: {video_count + community_count}")

if video_count:
    v_min = conn.execute("SELECT MIN(date_upload) FROM videos").fetchone()[0]
    v_max = conn.execute("SELECT MAX(date_upload) FROM videos").fetchone()[0]
    print(f"\nVideos date range: {v_min} ~ {v_max}")

if community_count:
    c_min = conn.execute("SELECT MIN(date_publish) FROM community").fetchone()[0]
    c_max = conn.execute("SELECT MAX(date_publish) FROM community").fetchone()[0]
    print(f"Community date range: {c_min} ~ {c_max}")

conn.close()

In [ ]:
# ── 최근 비디오 ───────────────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT video_id, date_upload, title FROM videos ORDER BY date_upload DESC LIMIT 10")
for row in cur:
    print(f"{row[1]}  [{row[0]}] {row[2]}")
conn.close()

In [ ]:
# ── 최근 커뮤니티 게시글 ──────────────────────────────────────────────────────
conn = sqlite3.connect(DB_PATH)
cur = conn.execute(
    "SELECT post_id, date_publish, substr(content_text, 1, 100) FROM community ORDER BY date_publish DESC LIMIT 10")
for row in cur:
    print(f"{row[1]}  [{row[0][:12]}] {row[2]}")
conn.close()